### ODELIA Metadata Generation
Description: This notebook merges the annotation files (`annotation.csv`) from the different institutions (CAM, MHA, RUMC, UKA) to create a centralized metadata file used by the training and inference scripts.
To run the scripts in this repository, generate this file locally and place it in the `Log_and_helper_files/` folder.

In [ ]:
# Merge all the annotations from the different datasets (excluding RHS since we don't have labels for this one)
# in one dataframe to be used for the training

import os
import pandas as pd

# Path configuration
BASE_ROOT = "/cluster/projects/vc/courses/TDT17/mic/ODELIA2025"
DATA_DIR = os.path.join(BASE_ROOT, "data")
# List of institutions with public labels
PROVIDERS = ["CAM", "MHA", "RUMC", "UKA"] 

all_annotations = []

for provider in PROVIDERS:
    # Path to the annotation.csv file of each center
    csv_path = os.path.join(DATA_DIR, provider, "metadata_unilateral", "annotation.csv")
    
    if os.path.exists(csv_path):
        # Read and add the institution to keep track of the source
        df_prov = pd.read_csv(csv_path)
        df_prov['Institution'] = provider 
        all_annotations.append(df_prov)
        print(f"✅ {provider} : {len(df_prov)} patients added.")
    else:
        print(f"⚠️ {provider} : File not found.")

# Merge all DataFrames
df_final = pd.concat(all_annotations, ignore_index=True)

# Join with the split file (to have Train/Val/Test columns)
split_path = os.path.join(BASE_ROOT, "split_unilateral.csv")
if os.path.exists(split_path):
    # 1. Merge the annotations from the 4 centers
    df_final = pd.concat(all_annotations, ignore_index=True)

    # 2. Keeping a unique row per UID with its label
    df_final = df_final.drop_duplicates(subset=['UID'])

    output_path = "../Log_and_helper_files/odelia_merged_metadata.csv"

    df_final.to_csv(output_path, index=False)


print("\n--- SUMMARY OF THE MERGED DATASET ---")
print(f"Total number of breasts : {len(df_final)}")
if 'Lesion' in df_final.columns:
    print("\nClass distribution (0:Normal, 1:Benign, 2:Malignant) :")
    print(df_final['Lesion'].value_counts())


✅ CAM : 374 patients ajoutés.
✅ MHA : 72 patients ajoutés.
✅ RUMC : 10 patients ajoutés.
✅ UKA : 22 patients ajoutés.

--- RÉSUMÉ DU DATASET FUSIONNÉ ---
Nombre total de seins : 478

Répartition des classes (0:Normal, 1:Bénin, 2:Malin) :
Lesion
0    334
2     96
1     48
Name: count, dtype: int64
